# Actividad 15v6 - GM v4 — loss ponderada por shocks (shock-weighted loss)
**Autor:** Fabrizio Sanchez Saravia - UPeU Juliaca

Identico a GM v3 (corpus 600 noticias, MAE=0.0645) salvo en la funcion de
perdida: GM v4 usa una MSE ponderada que penaliza 3x los meses con shock
(variacion > 0.3 en escala estandarizada de la produccion).

| Mejora | Descripcion |
|--------|-------------|
| M1 | nlp_index = avg_sentiment x log(n_noticias+1) |
| M2 | nlp_index_lag1 - lag 1 mes |
| M3 | Dropout=0.5 en rama NLP |
| M4 | PCA 95% varianza |
| F3 | Loss ponderada por shocks (peso 3x si variacion > 0.3 std) |


In [1]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f'GPU: {gpus[0].name}')
else:
    print('CPU mode')
print(f'TF: {tf.__version__}')
PROJECT_ROOT = Path('../..')
DATA_PATH   = PROJECT_ROOT / 'data/processed/master_dataset_fase2_multivariado.csv'
NLP_PATH    = PROJECT_ROOT / 'notebooks/fase2/output/01_nlp_sentimiento/sentimiento_mensual_v2.csv'
GE_METRICAS = PROJECT_ROOT / 'resultados/ge/ge_metricas.json'
GE_PRED     = PROJECT_ROOT / 'resultados/ge/ge_predicciones.csv'
OUT_DIR     = PROJECT_ROOT / 'resultados/gm_v4'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'DATA_PATH ok: {DATA_PATH.exists()}')
print(f'NLP_PATH  ok: {NLP_PATH.exists()}')

I0000 00:00:1781270223.288201   14331 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


GPU: /physical_device:GPU:0
TF: 2.21.0
DATA_PATH ok: True
NLP_PATH  ok: True


In [2]:
df_raw = pd.read_csv(DATA_PATH, parse_dates=['fecha_evento'])
print(f'Raw: {df_raw.shape}')
df_master = df_raw.groupby('fecha_evento').mean(numeric_only=True).reset_index()
df_master = df_master.sort_values('fecha_evento').reset_index(drop=True)
print(f'Agregado: {df_master.shape}')
print(f'Rango: {df_master["fecha_evento"].min().date()} -> {df_master["fecha_evento"].max().date()}')

Raw: (5880, 24)
Agregado: (56, 22)
Rango: 2021-01-01 -> 2025-08-01


In [3]:
df_nlp = pd.read_csv(NLP_PATH, encoding='utf-8-sig')
print(f'Columnas NLP: {df_nlp.columns.tolist()}')
fc = [c for c in df_nlp.columns if any(k in c.lower() for k in ['fecha','periodo','mes','month'])][0]
df_nlp = df_nlp.rename(columns={fc: 'fecha_evento'})
df_nlp['fecha_evento'] = pd.to_datetime(df_nlp['fecha_evento'])
df_nlp = df_nlp.sort_values('fecha_evento').reset_index(drop=True)
print(f'NLP: {df_nlp.shape}')
print(df_nlp['avg_sentiment'].describe())
print(df_nlp['n_noticias_beto'].describe())

Columnas NLP: ['fecha_evento', 'avg_sentiment', 'n_noticias_beto', 'n_positivas', 'n_negativas', 'n_neutrales']
NLP: (60, 6)
count    60.000000
mean     -0.049475
std       0.191534
min      -0.578000
25%      -0.188275
50%      -0.011300
75%       0.087400
max       0.293200
Name: avg_sentiment, dtype: float64
count    60.000000
mean     10.983333
std       5.488020
min       1.000000
25%       7.000000
50%      10.000000
75%      13.250000
max      28.000000
Name: n_noticias_beto, dtype: float64


In [4]:
df_nlp['nlp_index']      = df_nlp['avg_sentiment'] * np.log1p(df_nlp['n_noticias_beto'])
df_nlp['nlp_index_lag1'] = df_nlp['nlp_index'].shift(1).fillna(0)
print(df_nlp[['fecha_evento','avg_sentiment','n_noticias_beto','nlp_index','nlp_index_lag1']].head(10).to_string())
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
axes[0,0].plot(df_nlp['fecha_evento'], df_nlp['avg_sentiment'], color='steelblue', lw=1.5)
axes[0,0].axhline(0, color='red', ls='--', alpha=0.5)
axes[0,0].set_title('avg_sentiment crudo')
axes[0,0].grid(alpha=0.3)
axes[0,1].bar(df_nlp['fecha_evento'], df_nlp['n_noticias_beto'], color='orange', alpha=0.7)
axes[0,1].set_title('n_noticias_beto crudo')
axes[0,1].grid(alpha=0.3)
axes[1,0].plot(df_nlp['fecha_evento'], df_nlp['nlp_index'], color='darkgreen', lw=1.5)
axes[1,0].axhline(0, color='red', ls='--', alpha=0.5)
axes[1,0].set_title('nlp_index M1')
axes[1,0].grid(alpha=0.3)
axes[1,1].plot(df_nlp['fecha_evento'], df_nlp['nlp_index'], color='darkgreen', lw=1.5, label='t')
axes[1,1].plot(df_nlp['fecha_evento'], df_nlp['nlp_index_lag1'], color='purple', lw=1.5, ls='--', label='lag1')
axes[1,1].legend()
axes[1,1].set_title('nlp_index vs lag-1 M2')
axes[1,1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'nlp_features_engineering.png', dpi=150, bbox_inches='tight')
plt.close()
print('Grafico NLP guardado')

  fecha_evento  avg_sentiment  n_noticias_beto  nlp_index  nlp_index_lag1
0   2021-01-01         0.2932                7   0.609692        0.000000
1   2021-02-01        -0.0240                6  -0.046702        0.609692
2   2021-03-01         0.0870                5   0.155883       -0.046702
3   2021-04-01         0.0631                5   0.113060        0.155883
4   2021-05-01         0.1336                3   0.185209        0.113060
5   2021-06-01         0.2246                1   0.155681        0.185209
6   2021-07-01        -0.0900                4  -0.144849        0.155681
7   2021-08-01        -0.1830                2  -0.201046       -0.144849
8   2021-09-01        -0.0080                6  -0.015567       -0.201046
9   2021-10-01         0.0587               10   0.140756       -0.015567


Grafico NLP guardado


In [5]:
df = df_master.merge(df_nlp[['fecha_evento','nlp_index','nlp_index_lag1']], on='fecha_evento', how='left')
df['nlp_index']      = df['nlp_index'].fillna(0)
df['nlp_index_lag1'] = df['nlp_index_lag1'].fillna(0)
df = df.sort_values('fecha_evento').reset_index(drop=True)
TARGET = 'produccion_t'
META   = ['fecha_evento', TARGET]
STRUCT = [c for c in df_master.columns if c not in META]
NLP_F  = ['nlp_index', 'nlp_index_lag1']
print(f'Dataset fusionado: {df.shape}')
print(f'Struct ({len(STRUCT)}): {STRUCT}')
print(f'NLP: {NLP_F}')

Dataset fusionado: (56, 24)
Struct (20): ['precio_chacra_kg', 'num_emergencias', 'total_afectados', 'hectareas_cultivo_perdidas', 'ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'QV2M', 'RH2M', 'T2M', 'T2M_MAX', 'T2M_MIN', 'WS2M', 'lat', 'lon', 'month_sin', 'month_cos', 'mes_num', 'trimestre_num', 'trimestre_sin', 'trimestre_cos']
NLP: ['nlp_index', 'nlp_index_lag1']


In [6]:
TIMESTEPS = 6
n_total = len(df)
n_train = int(n_total * 0.80)
n_test  = n_total - n_train
df_train = df.iloc[:n_train].copy()
df_test  = df.iloc[n_train:].copy()
print(f'Train: {n_train} | {df_train["fecha_evento"].min().date()} -> {df_train["fecha_evento"].max().date()}')
print(f'Test:  {n_test}  | {df_test["fecha_evento"].min().date()} -> {df_test["fecha_evento"].max().date()}')
scaler_s = StandardScaler()
scaler_n = StandardScaler()
scaler_y = StandardScaler()
Xs_tr_raw = scaler_s.fit_transform(df_train[STRUCT])
Xs_te_raw = scaler_s.transform(df_test[STRUCT])
Xn_tr_raw = scaler_n.fit_transform(df_train[NLP_F])
Xn_te_raw = scaler_n.transform(df_test[NLP_F])
y_tr_sc   = scaler_y.fit_transform(df_train[[TARGET]])
y_te_sc   = scaler_y.transform(df_test[[TARGET]])
print('Escalado OK')

Train: 44 | 2021-01-01 -> 2024-08-01
Test:  12  | 2024-09-01 -> 2025-08-01
Escalado OK


In [7]:
pca = PCA(n_components=0.95, random_state=SEED)
Xs_tr = pca.fit_transform(Xs_tr_raw)
Xs_te = pca.transform(Xs_te_raw)
n_comp   = pca.n_components_
var_acum = np.cumsum(pca.explained_variance_ratio_)
print(f'PCA: {Xs_tr_raw.shape[1]} -> {n_comp} componentes')
for i,(v,a) in enumerate(zip(pca.explained_variance_ratio_, var_acum)):
    print(f'  PC{i+1}: {v:.3f} acum={a:.3f}')
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(range(1,n_comp+1), pca.explained_variance_ratio_, color='steelblue', alpha=0.7)
ax2 = ax.twinx()
ax2.plot(range(1,n_comp+1), var_acum, 'ro-', lw=2)
ax2.axhline(0.95, color='red', ls='--', alpha=0.5)
ax.set_title(f'PCA M4: {Xs_tr_raw.shape[1]} -> {n_comp} componentes')
plt.tight_layout()
plt.savefig(OUT_DIR / 'pca_varianza_explicada.png', dpi=150, bbox_inches='tight')
plt.close()
print('Grafico PCA guardado')

PCA: 20 -> 8 componentes
  PC1: 0.426 acum=0.426
  PC2: 0.235 acum=0.661
  PC3: 0.125 acum=0.786
  PC4: 0.062 acum=0.848
  PC5: 0.044 acum=0.893
  PC6: 0.031 acum=0.924
  PC7: 0.025 acum=0.949
  PC8: 0.016 acum=0.965
Grafico PCA guardado


In [8]:
def make_seq(Xs, Xn, y, ts):
    a, b, c = [], [], []
    for i in range(ts, len(Xs)):
        a.append(Xs[i-ts:i])
        b.append(Xn[i-ts:i])
        c.append(y[i])
    return np.array(a), np.array(b), np.array(c)

Xs_seq_tr, Xn_seq_tr, y_seq_tr = make_seq(Xs_tr, Xn_tr_raw, y_tr_sc, TIMESTEPS)
Xs_seq_te, Xn_seq_te, y_seq_te = make_seq(Xs_te, Xn_te_raw, y_te_sc, TIMESTEPS)
print(f'Xs_train: {Xs_seq_tr.shape}')
print(f'Xn_train: {Xn_seq_tr.shape}')
print(f'Xs_test:  {Xs_seq_te.shape}')
print(f'Xn_test:  {Xn_seq_te.shape}')

Xs_train: (38, 6, 8)
Xn_train: (38, 6, 2)
Xs_test:  (6, 6, 8)
Xn_test:  (6, 6, 2)


In [9]:
def build_gm_v4(ss, ns, units=64, drs=0.2, drn=0.5):
    inp_s = layers.Input(shape=ss, name='inp_struct')
    h  = layers.LSTM(units, return_sequences=True)(inp_s)
    sc = layers.Dense(1, activation='tanh')(h)
    sw = layers.Softmax(axis=1)(sc)
    ca = layers.Multiply()([h, sw])
    ca = layers.Lambda(lambda x: tf.reduce_sum(x, axis=1))(ca)
    ca = layers.Dropout(drs)(ca)
    inp_n = layers.Input(shape=ns, name='inp_nlp')
    cb = layers.LSTM(16, return_sequences=False)(inp_n)
    cb = layers.Dropout(drn, name='dropout_nlp_M3')(cb)
    mg = layers.Concatenate()([ca, cb])
    x  = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(mg)
    x  = layers.Dropout(0.2)(x)
    x  = layers.Dense(16, activation='relu')(x)
    out= layers.Dense(1)(x)
    return Model(inputs=[inp_s, inp_n], outputs=out, name='GM_v4')

ss = (Xs_seq_tr.shape[1], Xs_seq_tr.shape[2])
ns = (Xn_seq_tr.shape[1], Xn_seq_tr.shape[2])
model = build_gm_v4(ss, ns)
model.summary()
print(f'struct={ss} nlp={ns} | M3: Dropout NLP=0.5')

I0000 00:00:1781270225.666081   14331 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9709 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:07:00.0, compute capability: 8.6


Model: "GM_v4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ inp_struct          │ (None, 6, 8)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 6, 64)     │     18,688 │ inp_struct[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 6, 1)      │         65 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ softmax (Softmax)   │ (None, 6, 1)      │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 6, 64)     │          0 │ lstm[0][0],       │
│                     │                   │            │ softmax[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_nlp             │ (None, 6, 2)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 64)        │          0 │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 16)        │      1,216 │ inp_nlp[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_nlp_M3      │ (None, 16)        │          0 │ lstm_1[0][0]      │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 80)        │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dropout_nlp_M3[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      2,592 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 16)        │        528 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         17 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,106 (90.26 KB)

 Trainable params: 23,106 (90.26 KB)

 Non-trainable params: 0 (0.00 B)

struct=(6, 8) nlp=(6, 2) | M3: Dropout NLP=0.5


In [10]:
def shock_weighted_loss(y_true, y_pred):
    """
    MSE con peso mayor en meses de shock.
    Shocks = variacion > umbral en produccion.
    Peso 3.0 en shocks, 1.0 en meses normales.
    """
    error = tf.square(y_true - y_pred)
    # Detectar shocks: cambio absoluto > 0.3 (escala z-score)
    diff = tf.abs(y_true - tf.roll(y_true, shift=1, axis=0))
    es_shock_w = tf.cast(diff > 0.3, tf.float32)
    peso = 1.0 + 2.0 * es_shock_w  # 1.0 normal, 3.0 shock
    return tf.reduce_mean(peso * error)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=shock_weighted_loss,
    metrics=['mae']
)
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1)
]
print('Entrenando GM v4...')
history = model.fit(
    [Xs_seq_tr, Xn_seq_tr], y_seq_tr,
    epochs=200, batch_size=8, validation_split=0.2,
    callbacks=callbacks, shuffle=False, verbose=1
)

Entrenando GM v4...
Epoch 1/200


I0000 00:00:1781270228.358131   14418 cuda_dnn.cc:461] Loaded cuDNN version 92200


1/4 ━━━━━━━━━━━━━━━━━━━━ 8s 3s/step - loss: 1.2979 - mae: 0.6427

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 1.6351 - mae: 0.7299

4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 98ms/step - loss: 2.1829 - mae: 0.7926 - val_loss: 3.7642 - val_mae: 1.1504 - learning_rate: 0.0010


...

1/4 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 0.1173 - mae: 0.2524

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.1051 - mae: 0.2131

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.1175 - mae: 0.1930 - val_loss: 2.1263 - val_mae: 0.9318 - learning_rate: 2.5000e-04


Epoch 48: early stopping


Restoring model weights from the end of the best epoch: 33.


In [11]:
best_epoch    = int(np.argmin(history.history['val_loss'])) + 1
best_val_loss = float(min(history.history['val_loss']))
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].plot(history.history['loss'],     label='Train', color='steelblue')
axes[0].plot(history.history['val_loss'], label='Val',   color='orange')
axes[0].set_title('Loss MSE')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].plot(history.history['mae'],     label='Train', color='steelblue')
axes[1].plot(history.history['val_mae'], label='Val',   color='orange')
axes[1].set_title('MAE')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.suptitle(f'GM v4 | best_epoch={best_epoch} | val_loss={best_val_loss:.4f}', fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'gm_v4_training_curves.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'Best epoch={best_epoch} val_loss={best_val_loss:.4f}')

Best epoch=33 val_loss=1.9961


In [12]:
y_pred_sc = model.predict([Xs_seq_te, Xn_seq_te], verbose=0)
y_pred = scaler_y.inverse_transform(y_pred_sc).flatten()
y_true = scaler_y.inverse_transform(y_seq_te).flatten()
mae  = float(mean_absolute_error(y_true, y_pred))
rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
r2   = float(r2_score(y_true, y_pred))
mape = float(np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-8))) * 100)
if GE_METRICAS.exists():
    with open(GE_METRICAS) as f:
        ge = json.load(f)
    ge_mae  = ge.get('mae',  ge.get('MAE',  0.0673))
    ge_rmse = ge.get('rmse', ge.get('RMSE', 0.0698))
    ge_r2   = ge.get('r2',   ge.get('R2',   -2.34))
else:
    ge_mae, ge_rmse, ge_r2 = 0.0673, 0.0698, -2.34
gm_orig = {'mae':0.0981,'rmse':0.1007,'r2':-5.96}
gm_v2 = {'mae': 0.0646, 'rmse': 0.0771, 'r2': -9.86}  # referencia: corpus original 528 noticias
gm_v3 = {'mae': 0.0645, 'rmse': 0.0709, 'r2': -8.18}  # referencia: corpus ampliado 600 noticias
print('=' * 70)
print('COMPARATIVA FINAL')
print('=' * 70)
print(f"{'Metrica':<10} {'GE':>10} {'GM orig':>10} {'GM v2':>10} {'GM v3':>10} {'GM v4':>10}")
print(f"{'MAE':<10} {ge_mae:>10.4f} {gm_orig['mae']:>10.4f} {gm_v2['mae']:>10.4f} {gm_v3['mae']:>10.4f} {mae:>10.4f}")
print(f"{'RMSE':<10} {ge_rmse:>10.4f} {gm_orig['rmse']:>10.4f} {gm_v2['rmse']:>10.4f} {gm_v3['rmse']:>10.4f} {rmse:>10.4f}")
print(f"{'R2':<10} {ge_r2:>10.4f} {gm_orig['r2']:>10.4f} {gm_v2['r2']:>10.4f} {gm_v3['r2']:>10.4f} {r2:>10.4f}")
print(f"{'MAPE%':<10} {'N/A':>10} {'365.0':>10} {'N/A':>10} {'N/A':>10} {mape:>10.1f}")
print('=' * 70)
if mae < ge_mae:
    print(f'SUPERA GE: {(ge_mae-mae)/ge_mae*100:.1f}% mejor')
elif mae < gm_orig['mae']:
    print(f'Mejora sobre GM orig: {(gm_orig["mae"]-mae)/gm_orig["mae"]*100:.1f}%')
    print(f'Aun {(mae-ge_mae)/ge_mae*100:.1f}% peor que GE')
else:
    print('Sin mejora - desalineacion geografica NLP confirmada')
if mae < gm_v3['mae']:
    print(f"SUPERA GM v3: {(gm_v3['mae']-mae)/gm_v3['mae']*100:.1f}% mejor")
else:
    print(f"No supera GM v3: {mae:.4f} vs {gm_v3['mae']:.4f}")
if mae < gm_v2['mae']:
    print(f"SUPERA GM v2 (corpus original): {(gm_v2['mae']-mae)/gm_v2['mae']*100:.1f}% mejor")
else:
    print(f"No supera GM v2 (corpus original): {mae:.4f} vs {gm_v2['mae']:.4f}")

COMPARATIVA FINAL
Metrica            GE    GM orig      GM v2      GM v3      GM v4
MAE            0.0673     0.0981     0.0646     0.0645     0.0640
RMSE           0.0698     0.1007     0.0771     0.0709     0.0792
R2            -2.3447    -5.9600    -9.8600    -8.1800   -10.4516
MAPE%             N/A      365.0        N/A        N/A       96.8
SUPERA GE: 4.8% mejor
SUPERA GM v3: 0.7% mejor
SUPERA GM v2 (corpus original): 0.9% mejor


In [13]:
fechas_test = df['fecha_evento'].iloc[n_train + TIMESTEPS:].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(fechas_test, y_true, 'o-',  color='black',     lw=2, ms=5, label='Real')
ax.plot(fechas_test, y_pred, 's--', color='darkorange', lw=2, ms=5, label=f'GM v4 MAE={mae:.4f}')
if GE_PRED.exists():
    df_ge = pd.read_csv(GE_PRED)
    col_pred = [c for c in df_ge.columns if 'pred' in c.lower()]
    if col_pred:
        n_ov = min(len(fechas_test), len(df_ge))
        ax.plot(fechas_test[:n_ov], df_ge[col_pred[0]].values[:n_ov],
                '^:', color='steelblue', lw=1.5, ms=5, alpha=0.7, label='GE 0.0673')
ax.set_title('GM v4 - Predicciones vs Real', fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Produccion (media provincial)')
ax.legend()
ax.grid(alpha=0.3)
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(OUT_DIR / 'gm_v4_predicciones_vs_real.png', dpi=150, bbox_inches='tight')
plt.close()
print('Grafico predicciones guardado')

Grafico predicciones guardado


In [14]:
print('Ablation: sin lag M2...')
sc_nl = StandardScaler()
Xn_tr_nl = sc_nl.fit_transform(df_train[['nlp_index']])
Xn_te_nl = sc_nl.transform(df_test[['nlp_index']])
_, Xn_seq_tr_nl, _ = make_seq(Xs_tr, Xn_tr_nl, y_tr_sc, TIMESTEPS)
_, Xn_seq_te_nl, _ = make_seq(Xs_te, Xn_te_nl, y_te_sc, TIMESTEPS)
m_nl = build_gm_v4((Xs_seq_tr.shape[1], Xs_seq_tr.shape[2]),
                   (Xn_seq_tr_nl.shape[1], Xn_seq_tr_nl.shape[2]))
m_nl.compile(optimizer=keras.optimizers.Adam(1e-3), loss=shock_weighted_loss, metrics=['mae'])
m_nl.fit([Xs_seq_tr, Xn_seq_tr_nl], y_seq_tr,
         epochs=200, batch_size=8, validation_split=0.2,
         callbacks=[EarlyStopping(monitor='val_loss', patience=15,
                                  restore_best_weights=True, verbose=0)],
         shuffle=False, verbose=0)
y_pred_nl = scaler_y.inverse_transform(
    m_nl.predict([Xs_seq_te, Xn_seq_te_nl], verbose=0)).flatten()
mae_nl  = float(mean_absolute_error(y_true, y_pred_nl))
rmse_nl = float(np.sqrt(mean_squared_error(y_true, y_pred_nl)))
r2_nl   = float(r2_score(y_true, y_pred_nl))
print('ABLATION')
print(f"{'GM original':<32} MAE=0.0981 RMSE=0.1007 R2=-5.96")
print(f"{'GM v2 corpus original':<32} MAE=0.0646 RMSE=0.0771 R2=-9.86")
print(f"{'GM v3 corpus600':<32} MAE=0.0645 RMSE=0.0709 R2=-8.18")
print(f"{'GM v4 sin lag M2':<32} MAE={mae_nl:.4f} RMSE={rmse_nl:.4f} R2={r2_nl:.4f}")
print(f"{'GM v4 completo (M1+M2+M3+M4+F3)':<32} MAE={mae:.4f} RMSE={rmse:.4f} R2={r2:.4f}")
print(f"{'GE baseline sin NLP':<32} MAE=0.0673 RMSE=0.0698 R2=-2.34")
delta = mae_nl - mae
print(f'Lag M2: {delta:+.4f} ({"mejoro" if delta>0 else "no aporto"})')

Ablation: sin lag M2...


ABLATION
GM original                      MAE=0.0981 RMSE=0.1007 R2=-5.96
GM v2 corpus original            MAE=0.0646 RMSE=0.0771 R2=-9.86
GM v3 corpus600                  MAE=0.0645 RMSE=0.0709 R2=-8.18
GM v4 sin lag M2                 MAE=0.0731 RMSE=0.0862 R2=-12.5574
GM v4 completo (M1+M2+M3+M4+F3)  MAE=0.0640 RMSE=0.0792 R2=-10.4516
GE baseline sin NLP              MAE=0.0673 RMSE=0.0698 R2=-2.34
Lag M2: +0.0091 (mejoro)


In [15]:
resultados = {
    'modelo': 'GM_v4_DualLSTM_NLP_ShockLoss',
    'mejoras': ['M1_nlp_index','M2_lag1','M3_dropout_nlp_0.5','M4_pca_95pct',
                'F3_shock_weighted_loss'],
    'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape,
    'n_train': n_train, 'n_test': n_test,
    'pca_components': int(n_comp),
    'best_val_loss': best_val_loss,
    'best_epoch': best_epoch,
    'timesteps': TIMESTEPS,
    'n_features_canal_a': len(STRUCT),
    'n_features_canal_b': len(NLP_F),
    'comparativa': {
        'GE_sin_NLP':      {'MAE': ge_mae,         'RMSE': ge_rmse,          'R2': ge_r2},
        'GM_original':     {'MAE': gm_orig['mae'], 'RMSE': gm_orig['rmse'],  'R2': gm_orig['r2']},
        'GM_v2_M1M2M3M4':  {'MAE': gm_v2['mae'],   'RMSE': gm_v2['rmse'],    'R2': gm_v2['r2']},
        'GM_v3_corpus600': {'MAE': gm_v3['mae'],   'RMSE': gm_v3['rmse'],    'R2': gm_v3['r2']},
        'XGBoost':         {'MAE': 0.0471,         'RMSE': 0.0542,          'R2': -1.02},
        'GM_v4':           {'MAE': mae,            'RMSE': rmse,            'R2': r2}
    },
    'ablation_M2': {
        'sin_lag': {'MAE': mae_nl, 'RMSE': rmse_nl, 'R2': r2_nl},
        'con_lag': {'MAE': mae,    'RMSE': rmse,    'R2': r2}
    }
}
with open(OUT_DIR / 'gm_v4_metricas.json', 'w') as f:
    json.dump(resultados, f, indent=2)
pd.DataFrame({'fecha': fechas_test.values, 'real': y_true, 'pred_gm_v4': y_pred}).to_csv(
    OUT_DIR / 'gm_v4_predicciones.csv', index=False)
model.save(OUT_DIR / 'gm_v4_model.keras')
print('Archivos en resultados/gm_v4/:')
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f.name}')
print()
print('RESUMEN EJECUTIVO')
print(f'GE:      MAE={ge_mae:.4f} RMSE={ge_rmse:.4f} R2={ge_r2:.4f}')
print(f'GM orig: MAE=0.0981  RMSE=0.1007  R2=-5.96')
print(f'GM v2:   MAE=0.0646  RMSE=0.0771  R2=-9.86')
print(f'GM v3:   MAE=0.0645  RMSE=0.0709  R2=-8.18')
print(f'GM v4:   MAE={mae:.4f} RMSE={rmse:.4f} R2={r2:.4f}')
print(f'Canal A (struct): {len(STRUCT)} features | Canal B (NLP): {len(NLP_F)} features | Total: {len(STRUCT)+len(NLP_F)}')
if mae < ge_mae:
    print('RESULTADO: NLP mejorado SUPERA al GE')
elif mae < 0.0981:
    print('RESULTADO: Mejora parcial sobre GM original')
else:
    print('RESULTADO: Desalineacion geografica NLP-target confirmada')

Archivos en resultados/gm_v4/:
  gm_v4_metricas.json
  gm_v4_model.keras
  gm_v4_predicciones.csv
  gm_v4_predicciones_vs_real.png
  gm_v4_training_curves.png
  nlp_features_engineering.png
  pca_varianza_explicada.png

RESUMEN EJECUTIVO
GE:      MAE=0.0673 RMSE=0.0698 R2=-2.3447
GM orig: MAE=0.0981  RMSE=0.1007  R2=-5.96
GM v2:   MAE=0.0646  RMSE=0.0771  R2=-9.86
GM v3:   MAE=0.0645  RMSE=0.0709  R2=-8.18
GM v4:   MAE=0.0640 RMSE=0.0792 R2=-10.4516
Canal A (struct): 20 features | Canal B (NLP): 2 features | Total: 22
RESULTADO: NLP mejorado SUPERA al GE
